# Linear Regression Example with `rice_ml`

This notebook shows how to use the custom `LinearRegression` class in your `rice_ml` package on a real regression dataset.

This version is inspired by the Boston Housing example notebook structure, but it is adapted to your repository layout and your current implementation in:

`src/rice_ml/supervised_ml/linear_regression.py`

## Notebook goals

We will:

- load the Boston Housing dataset from the UCI repository
- inspect the data with a small amount of EDA
- split the data into training and testing sets
- standardize the predictors
- fit your custom `LinearRegression` model
- evaluate the model with R², MSE, RMSE, and MAE
- visualize predictions and residuals


---

## 1. Imports and project path setup

The next cell adds your local `src/` folder to Python's path, so the notebook can import `rice_ml` when it is placed inside:

`examples/Supervised_learning/linear_regression.ipynb`


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add the repository's src folder to sys.path
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.append(str(project_root / "src"))

from rice_ml.supervised_ml.linear_regression import LinearRegression


---

## 2. Load the dataset

We use the classic Boston Housing dataset from the UCI Machine Learning Repository.

- The predictors describe neighborhood and housing characteristics
- The response variable is `MEDV`, the median home value


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data"

column_names = [
    "CRIM", "ZN", "INDUS", "CHAS", "NOX",
    "RM", "AGE", "DIS", "RAD", "TAX",
    "PTRATIO", "B", "LSTAT", "MEDV"
]

df = pd.read_csv(url, sep=r"\s+", header=None, names=column_names)
df.head()


### Quick data check


In [ ]:
print("Shape:", df.shape)
print("\nMissing values by column:")
print(df.isna().sum())


---

## 3. Exploratory data analysis

We first look at summary statistics and then visualize the target variable.


In [ ]:
df.describe().T


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["MEDV"], bins=25, edgecolor="black")
plt.xlabel("MEDV")
plt.ylabel("Frequency")
plt.title("Distribution of Median Home Value")
plt.show()


### Correlation with the target

This helps us see which variables have stronger linear relationships with `MEDV`.


In [ ]:
correlations = df.corr(numeric_only=True)["MEDV"].sort_values(ascending=False)
correlations


In [ ]:
plt.figure(figsize=(8, 5))
correlations.drop("MEDV").plot(kind="bar")
plt.ylabel("Correlation with MEDV")
plt.title("Feature Correlations with Target")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


---

## 4. Train-test split and standardization

Because your current repository may organize helper utilities differently, this notebook uses small local helper functions for splitting and scaling.

That keeps the notebook self-contained and easy to run.


In [ ]:
def train_test_split_numpy(X, y, test_size=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    indices = np.arange(len(X))
    rng.shuffle(indices)

    test_count = int(len(X) * test_size)
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0

    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    return X_train_scaled, X_test_scaled, mean, std


In [ ]:
X = df.drop(columns="MEDV").to_numpy(dtype=float)
y = df["MEDV"].to_numpy(dtype=float)

X_train, X_test, y_train, y_test = train_test_split_numpy(
    X, y, test_size=0.2, random_state=42
)

X_train_scaled, X_test_scaled, train_mean, train_std = standardize_train_test(
    X_train, X_test
)

print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)


---

## 5. Fit the custom linear regression model

Your implementation supports:

- ordinary least squares
- optional ridge regularization
- optional gradient descent

Here we start with the default closed-form OLS fit.


In [ ]:
model = LinearRegression(
    fit_intercept=True,
    regularization=0.0,
    use_gradient_descent=False
)

model.fit(X_train_scaled, y_train)
print("Intercept:", model.intercept_)
print("Coefficients:")
print(model.coef_)


---

## 6. Model evaluation


In [ ]:
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

print("Training R^2:", round(model.score(X_train_scaled, y_train), 4))
print("Testing R^2: ", round(model.score(X_test_scaled, y_test), 4))
print()
print("Testing MSE: ", round(model.mse(X_test_scaled, y_test), 4))
print("Testing RMSE:", round(model.rmse(X_test_scaled, y_test), 4))
print("Testing MAE: ", round(model.mae(X_test_scaled, y_test), 4))


### First few predictions


In [ ]:
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred_test,
    "Residual": y_test - y_pred_test
})

results.head(10)


---

## 7. Visualizations


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_test, alpha=0.7)
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.xlabel("Actual MEDV")
plt.ylabel("Predicted MEDV")
plt.title("Actual vs Predicted Values")
plt.show()


In [ ]:
model.plot_residuals(X_test_scaled, y_test)


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(y_test - y_pred_test, bins=20, edgecolor="black")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.show()


---

## 8. Coefficient summary

Because the predictors were standardized, the coefficient magnitudes can be compared more directly.


In [ ]:
coef_df = pd.DataFrame({
    "Feature": df.drop(columns="MEDV").columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", key=np.abs, ascending=False)

coef_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(coef_df["Feature"], coef_df["Coefficient"])
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.title("Linear Regression Coefficients")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


---

## 9. Optional comparison: gradient descent fit

Since your class also supports gradient descent, we can compare it with the closed-form solution.


In [ ]:
gd_model = LinearRegression(
    fit_intercept=True,
    regularization=0.0,
    use_gradient_descent=True,
    learning_rate=0.01,
    max_iter=5000,
    tol=1e-8
)

gd_model.fit(X_train_scaled, y_train)

print("Gradient Descent Test R^2:", round(gd_model.score(X_test_scaled, y_test), 4))
print("Gradient Descent Test RMSE:", round(gd_model.rmse(X_test_scaled, y_test), 4))


---

## 10. Conclusion

This notebook demonstrates a complete regression workflow using your custom `rice_ml` implementation:

- data loading
- exploratory analysis
- scaling
- model fitting
- evaluation
- visualization

It is a good example notebook to place in your repository under:

`examples/Supervised_learning/linear_regression.ipynb`
